# Grammar Scoring Engine

Each WAV recording gets a grammar score from 0 to 5. The training file has 769 labeled clips. The test file has 216 clips whose `label` values are placeholders, not targets.

Acoustic features are a proxy for how the recording sounds (level, pauses, spectrum). They are not a parse of the words, so they do not measure grammar directly. On this small set the models are a constant mean, ridge regression, and a shallow gradient-boosting regressor.

**Reported from this run, separately:**

- In-sample training RMSE, from the model fit on all 769 labels
- 5-fold cross-validated RMSE and Pearson correlation (mean and standard deviation)
- RMSE and Pearson correlation on a 20% holdout that is not used to pick the model

No test-set score is claimed.

## Rubric

- **1** Limited control of basic grammatical structures and sentence syntax.
- **2** Basic structures with frequent grammatical errors or incomplete sentences.
- **3** Reasonable grammatical ability with noticeable syntax or structure errors.
- **4** Strong grammatical control with occasional minor errors.
- **5** High grammatical accuracy and effective use of complex structures.

Thirty-seven training scores are 0, outside the 1–5 rubric. They stay in the training set.

## 1. Load the data

Train audio is read only from `Dataset_Final/train/`. Test audio is read only from `Dataset_Final/test/`. Shared filenames are different files. The test `label` column (`-1`) is discarded. `sample_submission.csv` is a column template, not the row list.

In [ ]:
from pathlib import Path
import json
import os

ROOT = Path.cwd().resolve()
if not (ROOT / "Dataset_Final").exists():
    for candidate in [ROOT, *ROOT.parents]:
        if (candidate / "Dataset_Final").exists() and (candidate / "src").exists():
            os.chdir(candidate)
            ROOT = candidate.resolve()
            break
print("Project root:", ROOT)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.paths import (
    FEATURE_CACHE,
    FIGURE_DIR,
    METRICS_JSON,
    N_SPLITS,
    SAMPLE_SUBMISSION_CSV,
    SEED,
    SUBMISSION_CSV,
    TEST_CSV,
    ensure_output_dirs,
)
from src.data import SCORE_BINS, load_test_ids, load_train
from src.audio_features import (
    DURATION_FEATURES,
    EXCLUDED_FEATURES,
    load_or_build_feature_cache,
    model_feature_columns,
)
from src.evaluate import pearson, rmse
from src.models import HGBR_PARAMS
from src.pipeline import (
    boosting_factory,
    cross_validate,
    feature_matrix,
    fit_final,
    holdout_split,
    slice_metrics,
    stratified_cv,
)
from src.predict import verify_submission, write_submission

ensure_output_dirs()
CHOSEN_MODEL = "hgbr"
print("seed", SEED, "folds", N_SPLITS)
print("boosting parameters", HGBR_PARAMS)

In [ ]:
train_df = load_train()
test_df = load_test_ids()
raw_test = pd.read_csv(TEST_CSV)
sample_sub = pd.read_csv(SAMPLE_SUBMISSION_CSV)

print("train", len(train_df), "test", len(test_df), "sample_submission", len(sample_sub))
print("train file:", train_df["path"].iloc[0])
print("test file:", test_df["path"].iloc[0])
print("scores:", sorted(train_df["label"].unique()))
print("zeros:", int(train_df["is_zero"].sum()))
print(train_df["stratum"].value_counts().reindex(list(SCORE_BINS)).to_string())

assert len(train_df) == 769 and len(test_df) == 216
assert "label" not in test_df.columns
assert set(raw_test["label"].unique()) == {-1}
assert list(test_df["filename"]) == raw_test["filename"].astype(str).tolist()
assert all(Path(path).parent.name == "train" for path in train_df["path"])
assert all(Path(path).parent.name == "test" for path in test_df["path"])
assert list(sample_sub.columns) == ["filename", "label"] and len(sample_sub) != len(test_df)

shared = sorted(set(train_df["filename"]) & set(test_df["filename"]))
print("shared filenames, different directories:", len(shared))
for name in shared[:3]:
    train_path = Path(train_df.loc[train_df["filename"] == name, "path"].iloc[0]).resolve()
    test_path = Path(test_df.loc[test_df["filename"] == name, "path"].iloc[0]).resolve()
    assert train_path != test_path

zero_names = train_df.loc[train_df["is_zero"], "filename"]
print("zero files:", zero_names.min(), "to", zero_names.max())
print("zero files also in test.csv:", len(set(zero_names) & set(test_df["filename"])))

## 2. Extract features

Features are computed once per file and cached. Labels are not used. The cache is aligned to the CSV row order.

For each 16 kHz clip the extractor measures:

- level: RMS, peak, crest factor, silence fraction, and frame-energy percentiles
- speech activity: voiced-frame fraction and the rate of pauses of at least 0.2 s
- zero-crossing rate: mean and standard deviation
- 13 MFCCs: mean and standard deviation over time

The submitted model uses those 38 columns. It leaves out duration (train clips are mostly 60 s, test clips mostly 45 s, and duration barely tracks the score), spectral summaries that move with the zero-crossing rate, a high-frequency band that is always zero at 16 kHz, the clipping fraction, and absolute pause lengths.

In [ ]:
train_raw, test_raw = load_or_build_feature_cache(train_df, test_df, FEATURE_CACHE)
train_feat = train_df[["filename", "label", "stratum", "is_zero"]].merge(train_raw, on="filename", how="left")
test_feat = test_df[["filename"]].merge(test_raw, on="filename", how="left")
columns = model_feature_columns(False)
assert train_feat["filename"].tolist() == train_df["filename"].tolist()
assert test_feat["filename"].tolist() == test_df["filename"].tolist()
assert train_feat[columns].notna().all().all()
assert test_feat[columns].notna().all().all()
print("model features:", len(columns))
print(", ".join(columns))
print("left out:", ", ".join(DURATION_FEATURES + EXCLUDED_FEATURES))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
score_order = [0.0, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
counts = train_df["label"].value_counts().reindex(score_order).fillna(0)
axes[0].bar([str(score) for score in score_order], counts.to_numpy(), color="#4C78A8")
axes[0].set_title("Training scores (n=769)")
axes[0].set_xlabel("Grammar score")
axes[0].set_ylabel("Recordings")
axes[1].hist(train_feat["duration_sec"], bins=20, alpha=0.7, label="Train", color="#4C78A8")
axes[1].hist(test_feat["duration_sec"], bins=20, alpha=0.7, label="Test", color="#F58518")
axes[1].set_title("Duration: train vs test")
axes[1].set_xlabel("Duration (s)")
axes[1].set_ylabel("Recordings")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "score_and_duration.png", dpi=120)
plt.show()
print("train duration median", round(float(train_feat["duration_sec"].median()), 2))
print("test duration median", round(float(test_feat["duration_sec"].median()), 2))
print("Pearson(duration, score)", round(pearson(train_feat["label"], train_feat["duration_sec"]), 3))

## 3. Compare models

Every row uses the same stratified folds (seed 42, bins `0 | 1–2 | 2.5–3.5 | 4–5`). Ridge scaling and the boosting model are fit on the training side of each fold only.

`mean_train_rmse` is the average error inside those training folds. It is optimistic. The selection numbers are validation RMSE and Pearson correlation. A gap smaller than the fold standard deviation is not treated as a real improvement, and a small cross-validation change is not treated as a guaranteed leaderboard change.

The extra boosting rows are the only alternatives tried in this pass: stronger L2, larger leaves, shallower trees, and the same model with duration added back.

In [ ]:
y = train_feat["label"].to_numpy(dtype=np.float64)
strata = train_feat["stratum"].to_numpy()
zero_mask = train_feat["is_zero"].to_numpy()
X = feature_matrix(train_feat, columns)
X_duration = feature_matrix(train_feat, model_feature_columns(True))

candidates = [
    ("mean", stratified_cv(np.zeros((len(y), 1)), y, strata, None, n_splits=N_SPLITS, seed=SEED)),
    ("ridge", stratified_cv(X, y, strata, "ridge", n_splits=N_SPLITS, seed=SEED)),
    ("hgbr", stratified_cv(X, y, strata, "hgbr", n_splits=N_SPLITS, seed=SEED)),
    ("hgbr_l2_10", cross_validate(X, y, strata, boosting_factory(l2_regularization=10.0), n_splits=N_SPLITS, seed=SEED)),
    ("hgbr_leaf_40", cross_validate(X, y, strata, boosting_factory(min_samples_leaf=40), n_splits=N_SPLITS, seed=SEED)),
    ("hgbr_depth_2", cross_validate(X, y, strata, boosting_factory(max_depth=2, max_iter=300), n_splits=N_SPLITS, seed=SEED)),
    ("hgbr_with_duration", stratified_cv(X_duration, y, strata, "hgbr", n_splits=N_SPLITS, seed=SEED)),
]

rows = []
for name, result in candidates:
    nonzero = slice_metrics(y, result["oof_pred"], ~zero_mask)
    rows.append({
        "model": name,
        "mean_train_rmse": result["mean_train_rmse"],
        "mean_val_rmse": result["mean_val_rmse"],
        "std_val_rmse": result["std_val_rmse"],
        "mean_val_pearson": result["mean_val_pearson"],
        "std_val_pearson": result["std_val_pearson"],
        "nonzero_oof_rmse": nonzero["rmse"],
        "nonzero_oof_pearson": nonzero["pearson"],
    })
cv_table = pd.DataFrame(rows)
print(cv_table.round(4).to_string(index=False))

chosen = cv_table.loc[cv_table["model"] == CHOSEN_MODEL].iloc[0]
best_other = cv_table.loc[cv_table["model"] != CHOSEN_MODEL, "mean_val_rmse"].min()
print()
print("Chosen model:", CHOSEN_MODEL)
print(
    "validation RMSE", round(float(chosen["mean_val_rmse"]), 4),
    "+/-", round(float(chosen["std_val_rmse"]), 4),
)
print(
    "validation Pearson", round(float(chosen["mean_val_pearson"]), 4),
    "+/-", round(float(chosen["std_val_pearson"]), 4),
)
print("best other validation RMSE", round(float(best_other), 4))
print("Keeping the default boosting model: the other boosting settings are not better by a margin that exceeds fold-to-fold variation, and ridge is worse.")
chosen_cv = dict(candidates)[CHOSEN_MODEL]

## 4. Holdout check

This 20% split uses the same seed and the same score bins. It is drawn after the model choice above and is not used to change that choice.

In [ ]:
holdout = holdout_split(X, y, strata, CHOSEN_MODEL, test_size=0.2, seed=SEED)
print("holdout rows: train", holdout["n_train"], "validation", holdout["n_val"])
print("holdout training RMSE (fit on 80% only):", round(holdout["train_rmse"], 4))
print("holdout validation RMSE:", round(holdout["val_rmse"], 4))
print("holdout validation Pearson:", round(holdout["val_pearson"], 4))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(holdout["y_val"], holdout["pred_val"], alpha=0.7, color="#4C78A8")
axes[0].plot([0, 5], [0, 5], color="black", linewidth=1)
axes[0].set_xlabel("True score")
axes[0].set_ylabel("Predicted score")
axes[0].set_title("Holdout: predicted vs actual")
axes[0].set_xlim(-0.2, 5.2)
axes[0].set_ylim(-0.2, 5.2)
residuals = holdout["pred_val"] - holdout["y_val"]
axes[1].scatter(holdout["pred_val"], residuals, alpha=0.7, color="#F58518")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_xlabel("Predicted score")
axes[1].set_ylabel("Residual (predicted - true)")
axes[1].set_title("Holdout residuals")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "holdout_residuals.png", dpi=120)
plt.show()

## 5. Zero scores

The 37 zeros are easy to recognize (no pauses, high level, a flat spectrum), so the correlation on all 769 rows is higher than the correlation on scores 1–5. Test filenames do not include that block (`audio_5037.wav`–`audio_5073.wav`). The zeros stay in the final fit: dropping them changes validation only in the third digit, which is not a strong enough reason to discard labeled rows.

In [ ]:
oof = chosen_cv["oof_pred"]
print("out-of-fold, zeros kept in every training fold")
print("  all rows ", slice_metrics(y, oof, np.ones(len(y), dtype=bool)))
print("  zeros    ", slice_metrics(y, oof, zero_mask))
print("  non-zero ", slice_metrics(y, oof, ~zero_mask))

kept = ~zero_mask
without_zeros = stratified_cv(
    X[kept], y[kept], strata[kept], CHOSEN_MODEL, n_splits=N_SPLITS, seed=SEED
)
print("diagnostic refit on the 732 non-zero rows only (not the submitted model)")
print("  validation RMSE", round(without_zeros["mean_val_rmse"], 4), "+/-", round(without_zeros["std_val_rmse"], 4))
print("  validation Pearson", round(without_zeros["mean_val_pearson"], 4), "+/-", round(without_zeros["std_val_pearson"], 4))

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.hist(oof[zero_mask], bins=10, alpha=0.85, label="Predicted, true = 0", color="#E45756")
ax.hist(oof[~zero_mask], bins=15, alpha=0.5, label="Predicted, true > 0", color="#4C78A8")
ax.set_xlabel("Out-of-fold prediction")
ax.set_ylabel("Recordings")
ax.set_title("Zero-score clips are easy to separate")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "zero_oof.png", dpi=120)
plt.show()

## 6. Train the final model and write the submission

The model is fit on all 769 labels. The RMSE in this section is **in-sample training RMSE**: every training label was seen during the fit. The cross-validation and holdout numbers above are the generalization estimates.

The submission has one row per `test.csv` line, in that order, with columns `filename,label`. Scores are finite and clipped to `[0, 5]`.

In [ ]:
model, train_pred, train_rmse = fit_final(X, y, CHOSEN_MODEL)
mean_rmse = rmse(y, np.full_like(y, float(np.mean(y))))
test_pred = np.clip(model.predict(feature_matrix(test_feat, columns)), 0.0, 5.0)
print("IN-SAMPLE training RMSE, constant mean:", round(mean_rmse, 4))
print("IN-SAMPLE training RMSE, boosting model:", round(train_rmse, 4))
print("IN-SAMPLE training Pearson (optimistic):", round(pearson(y, train_pred), 4))

submission = write_submission(test_df["filename"].tolist(), test_pred, SUBMISSION_CSV)
verification = verify_submission(SUBMISSION_CSV)
assert verification["ok"], verification["issues"]
assert list(submission["filename"]) == raw_test["filename"].astype(str).tolist()
assert np.isfinite(submission["label"].to_numpy(dtype=float)).all()
print(verification)
print(submission.head())

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.hist(train_df["label"], bins=np.arange(-0.25, 5.76, 0.5), alpha=0.6, label="Train labels", color="#4C78A8")
ax.hist(test_pred, bins=np.arange(-0.25, 5.76, 0.5), alpha=0.6, label="Test predictions", color="#F58518")
ax.set_xlabel("Grammar score")
ax.set_ylabel("Recordings")
ax.set_title("Train labels and test predictions")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "pred_distribution.png", dpi=120)
plt.show()

In [ ]:
def to_py(obj):
    if isinstance(obj, dict):
        return {key: to_py(value) for key, value in obj.items()}
    if isinstance(obj, list):
        return [to_py(value) for value in obj]
    if isinstance(obj, (np.floating, float)):
        value = float(obj)
        return None if np.isnan(value) else value
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj

metrics = {
    "seed": SEED,
    "chosen_model": CHOSEN_MODEL,
    "hgbr_params": HGBR_PARAMS,
    "n_features": len(columns),
    "n_train": int(len(y)),
    "n_test": int(len(test_df)),
    "n_zeros_retained": int(zero_mask.sum()),
    "insample_mean_rmse": mean_rmse,
    "insample_train_rmse": train_rmse,
    "cv5_mean_train_rmse": float(chosen["mean_train_rmse"]),
    "cv5_mean_val_rmse": float(chosen["mean_val_rmse"]),
    "cv5_std_val_rmse": float(chosen["std_val_rmse"]),
    "cv5_mean_val_pearson": float(chosen["mean_val_pearson"]),
    "cv5_std_val_pearson": float(chosen["std_val_pearson"]),
    "holdout20_train_rmse": holdout["train_rmse"],
    "holdout20_val_rmse": holdout["val_rmse"],
    "holdout20_val_pearson": holdout["val_pearson"],
    "oof_nonzero_rmse": float(chosen["nonzero_oof_rmse"]),
    "oof_nonzero_pearson": float(chosen["nonzero_oof_pearson"]),
    "cv_table": cv_table.to_dict(orient="records"),
    "submission": verification,
}
METRICS_JSON.write_text(json.dumps(to_py(metrics), indent=2), encoding="utf-8")
print("IN-SAMPLE training RMSE:     ", round(train_rmse, 4))
print("5-fold validation RMSE:       ", round(float(chosen["mean_val_rmse"]), 4), "+/-", round(float(chosen["std_val_rmse"]), 4))
print("5-fold validation Pearson:    ", round(float(chosen["mean_val_pearson"]), 4), "+/-", round(float(chosen["std_val_pearson"]), 4))
print("20% holdout RMSE / Pearson:   ", round(holdout["val_rmse"], 4), round(holdout["val_pearson"], 4))
print("Saved", METRICS_JSON)

## Limitations

The score being predicted is grammatical quality. The inputs are acoustic summaries, so the model can track fluency and recording condition without knowing whether a sentence is grammatical.

Overall correlation is helped by the 37 zero-score recordings, which the model separates easily. On the other scores the out-of-fold correlation is lower, and that is the more relevant figure for a test set that does not contain those files. The zeros are still included in the fit.

Train and test clips differ in length and in level. Duration was checked and did not improve the cross-validation. Ridge is clearly worse than shallow boosting here. Changing the boosting penalty, leaf size, or depth did not beat the default settings by more than the spread across folds. That is why the default model is submitted. A difference of a few thousandths of RMSE on these folds would not be a reliable reason to expect a better hidden leaderboard score.

Speaker identity is unknown, so the same speaker can appear in more than one fold. Test labels were not used.